# 38. The Automation Investment Analysis Problem
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Goal
Formulate and solve the automation investment analysis as a multi-period stochastic programming model that captures uncertainty in technology outcomes, market conditions, and operational performance.

### Key assumptions
- Investment decisions are binary (select/don't select each project)
- Multiple scenarios represent different market conditions and technology outcomes
- Projects can be implemented gradually over time
- Synergy effects exist between compatible projects
- Budget constraint limits total investment

### Approach (step-by-step)
1. Define mathematical model with sets, parameters, and decision variables
2. Formulate objective function to maximize expected NPV
3. Add constraints for budget, implementation, and project compatibility
4. Solve using mixed-integer programming with pulp solver
5. Perform sensitivity analysis on key parameters

### What to look for in the results
- Optimal project selection under budget constraint
- Expected NPV calculation across scenarios
- Synergy effects between selected projects
- Implementation schedule over time periods

### Concrete example (from the source)
Three automation projects for a container terminal:
- Project 1: Autonomous crane system ($20M investment)
- Project 2: AI-powered yard management ($15M investment)
- Project 3: Integrated digital twin platform ($18M investment)

With a $50M budget constraint and three market scenarios:
- High Growth: 40% probability
- Base Case: 40% probability
- Low Growth: 20% probability

Expected annual benefits (in millions):
- Project 1: High: $8M, Base: $5M, Low: $2M
- Project 2: High: $6M, Base: $4M, Low: $2M
- Project 3: High: $7M, Base: $4M, Low: $1M

Synergy between Projects 1 and 2: 15% additional benefit when both selected.

### Visualization(s)
- Investment portfolio structure
- NPV breakdown by project and scenario
- Sensitivity analysis charts

### Why this Tier exists vs earlier Tiers
This is the foundational tier that provides the mathematical framework for automation investment analysis. It establishes the baseline optimization approach against which all other methods will be compared.

### Pros / Cons vs other approaches
**Pros:**
- Provides provably optimal solutions
- Handles uncertainty through scenario analysis
- Clear mathematical formulation
- Transparent decision logic

**Cons:**
- Computationally intensive for large instances
- Requires precise parameter estimation
- Limited scalability with many projects and scenarios
- Assumes rational decision-making

### When to use this Tier
- Small to medium-sized investment portfolios (≤ 20 projects)
- When optimality guarantees are required
- Regulatory or stakeholder requirements for transparent analysis
- Strategic decisions with high financial impact

In [1]:
# Import required libraries for mathematical optimization
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from dataclasses import dataclass
from typing import List, Dict, Tuple

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for automation investment analysis")

Libraries imported successfully for automation investment analysis


Libraries imported successfully for automation investment analysis


Libraries imported successfully for automation investment analysis


Libraries imported successfully for automation investment analysis


Libraries imported successfully for automation investment analysis


In [ ]:
@dataclass
class AutomationProject:
    """Represents an automation investment project"""
    id: int
    name: str
    cost: float  # Initial investment in millions
    benefits: Dict[str, float]  # Benefits per scenario (millions per year)
    implementation_periods: int = 5  # Number of periods for gradual implementation

@dataclass
class MarketScenario:
    """Represents a market scenario with probability"""
    name: str
    probability: float

@dataclass
class Synergy:
    """Represents synergy between two projects"""
    project1_id: int
    project2_id: int
    bonus_percentage: float

print("Data structures defined for investment analysis")

In [ ]:
# Define the concrete example from the source material
projects = [
    AutomationProject(
        id=1,
        name="Autonomous Crane System",
        cost=20.0,
        benefits={"High": 8.0, "Base": 5.0, "Low": 2.0}
    ),
    AutomationProject(
        id=2,
        name="AI-Powered Yard Management",
        cost=15.0,
        benefits={"High": 6.0, "Base": 4.0, "Low": 2.0}
    ),
    AutomationProject(
        id=3,
        name="Integrated Digital Twin Platform",
        cost=18.0,
        benefits={"High": 7.0, "Base": 4.0, "Low": 1.0}
    )
]

scenarios = [
    MarketScenario("High", 0.4),
    MarketScenario("Base", 0.4),
    MarketScenario("Low", 0.2)
]

synergies = [
    Synergy(1, 2, 0.15)  # 15% synergy between projects 1 and 2
]

# Problem parameters
budget = 50.0  # Total budget in millions
discount_rate = 0.10  # 10% discount rate
planning_horizon = 10  # 10 years

print(f"Defined {len(projects)} projects, {len(scenarios)} scenarios, and {len(synergies)} synergies")
print(f"Total budget: ${budget}M, Planning horizon: {planning_horizon} years")

In [ ]:
def calculate_npv(project: AutomationProject, scenario: MarketScenario, 
                 discount_rate: float, horizon: int) -> float:
    """Calculate NPV for a single project under a specific scenario"""
    annual_benefit = project.benefits[scenario.name]
    npv = -project.cost  # Initial investment
    
    # Add discounted benefits over planning horizon
    for year in range(1, horizon + 1):
        npv += annual_benefit / ((1 + discount_rate) ** year)
    
    return npv

def calculate_portfolio_npv(selected_projects: List[AutomationProject], 
                            scenarios: List[MarketScenario],
                            synergies: List[Synergy],
                            discount_rate: float, horizon: int) -> float:
    """Calculate expected NPV for a portfolio of projects"""
    expected_npv = 0.0
    
    for scenario in scenarios:
        scenario_npv = 0.0
        
        # Base NPV from individual projects
        for project in selected_projects:
            scenario_npv += calculate_npv(project, scenario, discount_rate, horizon)
        
        # Add synergy bonuses
        for synergy in synergies:
            project_ids = [p.id for p in selected_projects]
            if synergy.project1_id in project_ids and synergy.project2_id in project_ids:
                # Find the two projects
                proj1 = next(p for p in selected_projects if p.id == synergy.project1_id)
                proj2 = next(p for p in selected_projects if p.id == synergy.project2_id)
                
                # Calculate synergy bonus (15% of combined benefits)
                combined_benefit = proj1.benefits[scenario.name] + proj2.benefits[scenario.name]
                synergy_bonus = synergy.bonus_percentage * combined_benefit
                
                # Discount synergy benefits over horizon
                for year in range(1, horizon + 1):
                    scenario_npv += synergy_bonus / ((1 + discount_rate) ** year)
        
        expected_npv += scenario.probability * scenario_npv
    
    return expected_npv

print("NPV calculation functions defined")

In [ ]:
# Calculate individual project NPVs for reference
print("Individual Project Analysis:")
print("=" * 50)

for project in projects:
    print(f"\n{project.name} (${project.cost}M):")
    for scenario in scenarios:
        npv = calculate_npv(project, scenario, discount_rate, planning_horizon)
        print(f"  {scenario.name} Scenario: ${npv:.2f}M")
    
    # Expected NPV across scenarios
    expected_npv = sum(scenario.probability * 
                       calculate_npv(project, scenario, discount_rate, planning_horizon)
                       for scenario in scenarios)
    print(f"  Expected NPV: ${expected_npv:.2f}M")

In [ ]:
def solve_investment_optimization(projects: List[AutomationProject],
                                 scenarios: List[MarketScenario],
                                 synergies: List[Synergy],
                                 budget: float,
                                 discount_rate: float,
                                 horizon: int) -> Tuple[pulp.LpProblem, List[AutomationProject]]:
    """Solve the investment optimization problem using mixed-integer programming"""
    
    # Create the optimization problem
    prob = pulp.LpProblem("Automation_Investment_Optimization", pulp.LpMaximize)
    
    # Decision variables: x_i = 1 if project i is selected, 0 otherwise
    x = {}
    for project in projects:
        x[project.id] = pulp.LpVariable(f"x_{project.id}", cat='Binary')
    
    # Implementation level variables (simplified for this example)
    y = {}
    for project in projects:
        for t in range(horizon):
            y[project.id, t] = pulp.LpVariable(f"y_{project.id}_{t}", lowBound=0, upBound=1)
    
    # Objective function: Maximize expected NPV
    objective = 0
    
    # Add project contributions
    for scenario in scenarios:
        for project in projects:
            annual_benefit = project.benefits[scenario.name]
            for t in range(horizon):
                discounted_benefit = annual_benefit / ((1 + discount_rate) ** (t + 1))
                objective += scenario.probability * discounted_benefit * y[project.id, t]
        
        # Add synergy effects
        for synergy in synergies:
            proj1 = next(p for p in projects if p.id == synergy.project1_id)
            proj2 = next(p for p in projects if p.id == synergy.project2_id)
            
            combined_benefit = proj1.benefits[scenario.name] + proj2.benefits[scenario.name]
            synergy_bonus = synergy.bonus_percentage * combined_benefit
            
            for t in range(horizon):
                discounted_synergy = synergy_bonus / ((1 + discount_rate) ** (t + 1))
                objective += scenario.probability * discounted_synergy * x[synergy.project1_id] * x[synergy.project2_id]
    
    # Subtract investment costs
    for project in projects:
        objective -= project.cost * x[project.id]
    
    prob += objective
    
    # Constraints
    
    # Budget constraint
    prob += pulp.lpSum(project.cost * x[project.id] for project in projects) <= budget
    
    # Implementation requires selection
    for project in projects:
        for t in range(horizon):
            prob += y[project.id, t] <= x[project.id]
    
    # Gradual implementation (simplified - assume full implementation from year 1)
    for project in projects:
        for t in range(horizon):
            prob += y[project.id, t] == x[project.id]  # Full implementation
    
    # Solve the problem
    print("Solving optimization problem...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Extract selected projects
    selected_projects = []
    for project in projects:
        if pulp.value(x[project.id]) > 0.5:
            selected_projects.append(project)
    
    return prob, selected_projects

print("Optimization function defined")

In [ ]:
# Solve the optimization problem
prob, selected_projects = solve_investment_optimization(
    projects, scenarios, synergies, budget, discount_rate, planning_horizon
)

# Display results
print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)

print(f"\nProblem Status: {pulp.LpStatus[prob.status]}")
print(f"Objective Value (Expected NPV): ${pulp.value(prob.objective):.2f}M")

print(f"\nSelected Projects ({len(selected_projects)}):")
total_cost = 0
for project in selected_projects:
    print(f"  - {project.name}: ${project.cost}M")
    total_cost += project.cost

print(f"\nTotal Investment: ${total_cost}M")
print(f"Budget Utilization: {(total_cost/budget)*100:.1f}%")
print(f"Remaining Budget: ${budget - total_cost}M")

# Calculate detailed portfolio analysis
portfolio_npv = calculate_portfolio_npv(selected_projects, scenarios, synergies, discount_rate, planning_horizon)
print(f"\nPortfolio Expected NPV: ${portfolio_npv:.2f}M")

# Synergy analysis
print("\nSynergy Effects:")
for synergy in synergies:
    project_ids = [p.id for p in selected_projects]
    if synergy.project1_id in project_ids and synergy.project2_id in project_ids:
        proj1 = next(p for p in selected_projects if p.id == synergy.project1_id)
        proj2 = next(p for p in selected_projects if p.id == synergy.project2_id)
        print(f"  - {proj1.name} + {proj2.name}: {synergy.bonus_percentage*100}% synergy bonus")

In [ ]:
# Sensitivity analysis on budget
budget_range = np.arange(30, 81, 5)  # Budget from $30M to $80M
budget_results = []

for test_budget in budget_range:
    _, test_selected = solve_investment_optimization(
        projects, scenarios, synergies, test_budget, discount_rate, planning_horizon
    )
    
    total_cost = sum(p.cost for p in test_selected)
    portfolio_npv = calculate_portfolio_npv(test_selected, scenarios, synergies, discount_rate, planning_horizon)
    
    budget_results.append({
        'budget': test_budget,
        'projects': len(test_selected),
        'total_cost': total_cost,
        'npv': portfolio_npv
    })

# Create sensitivity analysis visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Budget vs NPV
budgets = [r['budget'] for r in budget_results]
npvs = [r['npv'] for r in budget_results]
ax1.plot(budgets, npvs, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Budget ($M)')
ax1.set_ylabel('Expected NPV ($M)')
ax1.set_title('Budget vs Expected NPV')
ax1.grid(True, alpha=0.3)

# Plot 2: Budget vs Number of Projects
project_counts = [r['projects'] for r in budget_results]
ax2.bar(budgets, project_counts, color='skyblue', alpha=0.7)
ax2.set_xlabel('Budget ($M)')
ax2.set_ylabel('Number of Projects')
ax2.set_title('Budget vs Number of Selected Projects')
ax2.grid(True, alpha=0.3)

# Plot 3: Project cost breakdown
project_names = [p.name for p in projects]
project_costs = [p.cost for p in projects]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
ax3.barh(project_names, project_costs, color=colors)
ax3.set_xlabel('Cost ($M)')
ax3.set_title('Individual Project Costs')
ax3.grid(True, alpha=0.3)

# Plot 4: Scenario analysis for optimal solution
scenario_data = []
for scenario in scenarios:
    scenario_npv = 0
    for project in selected_projects:
        scenario_npv += calculate_npv(project, scenario, discount_rate, planning_horizon)
    
    # Add synergies
    for synergy in synergies:
        project_ids = [p.id for p in selected_projects]
        if synergy.project1_id in project_ids and synergy.project2_id in project_ids:
            proj1 = next(p for p in selected_projects if p.id == synergy.project1_id)
            proj2 = next(p for p in selected_projects if p.id == synergy.project2_id)
            combined_benefit = proj1.benefits[scenario.name] + proj2.benefits[scenario.name]
            synergy_bonus = synergy.bonus_percentage * combined_benefit
            
            for year in range(1, planning_horizon + 1):
                scenario_npv += synergy_bonus / ((1 + discount_rate) ** year)
    
    scenario_data.append(scenario_npv)

scenario_names = [s.name for s in scenarios]
scenario_probs = [s.probability for s in scenarios]
ax4.pie(scenario_data, labels=[f"{name}\n${npv:.1f}M" for name, npv in zip(scenario_names, scenario_data)], 
        autopct='%1.1f%%', startangle=90)
ax4.set_title('NPV Distribution by Scenario\nfor Optimal Portfolio')

plt.tight_layout()
plt.show()

print("Sensitivity analysis completed and visualized")

In [ ]:
# Comprehensive results summary
print("\n" + "=" * 70)
print("COMPREHENSIVE INVESTMENT ANALYSIS SUMMARY")
print("=" * 70)

print(f"\nOptimal Portfolio Summary:")
print(f"- Total Investment: ${total_cost}M of ${budget}M budget")
print(f"- Budget Utilization: {(total_cost/budget)*100:.1f}%")
print(f"- Expected Portfolio NPV: ${portfolio_npv:.2f}M")
print(f"- Number of Projects: {len(selected_projects)}")

print(f"\nSelected Projects Detail:")
for i, project in enumerate(selected_projects, 1):
    project_npv = sum(scenario.probability * 
                     calculate_npv(project, scenario, discount_rate, planning_horizon)
                     for scenario in scenarios)
    print(f"{i}. {project.name}")
    print(f"   - Investment: ${project.cost}M")
    print(f"   - Expected NPV: ${project_npv:.2f}M")
    print(f"   - ROI: {(project_npv/project.cost - 1)*100:.1f}%")

print(f"\nKey Insights:")
print(f"- The optimization selects projects that maximize expected NPV while respecting budget constraints")
print(f"- Synergy effects between compatible projects create additional value")
print(f"- Scenario analysis accounts for market uncertainty and technology outcome variability")
print(f"- The mathematical formulation provides a provably optimal solution for the given parameters")

print(f"\nMathematical Model Summary:")
print(f"- Decision Variables: {len(projects)} binary selection variables")
print(f"- Constraints: Budget + Implementation requirements")
print(f"- Objective: Maximize expected NPV across {len(scenarios)} scenarios")
print(f"- Planning Horizon: {planning_horizon} years with {discount_rate*100}% discount rate")

print(f"\n" + "=" * 70)